# SportMetrics Platform
# Databricks Implementation
## Modern Data Stack · Delta Lake · Unity Catalog

**Auteur :** Yoan Charney Mboulou  
**Stack :** Databricks · Delta Lake · Unity Catalog · dbt Core  
**Repo original :** github.com/Yohan-Charney/Sports_Metrics  

---

### Contexte du projet

SportMetrics est une plateforme data end-to-end conçue pour 
le suivi de la performance et la prévention des blessures d'une 
équipe de basketball.

Ce notebook implémente l'architecture sur **Databricks Delta Lake** 
démontrant la portabilité du pipeline initialement développé 
sur BigQuery et Snowflake, sans modification de la logique métier.

---

### Décisions d'architecture

| Choix | Justification |
|---|---|
| **Delta Lake** | Time Travel natif · MERGE optimisé · Format ouvert |
| **Unity Catalog** | Gouvernance centralisée · Lineage · Tags RGPD |
| **Views en staging** | Pas de stockage redondant · Données toujours fraîches |
| **MERGE en mart** | Chargement incrémental · Préserve l'historique Delta |
| **CSV → Volume** | Ingestion reproductible depuis fichiers sources |

---

### Architecture des données

| Couche | Schéma | Type | Description |
|---|---|---|---|
| Ingestion | `/Volumes/.../files/` | Fichiers | CSV sources |
| Raw | `sport_metrics.raw` | Table Delta | Données brutes |
| Staging | `sport_metrics.staging` | View | Nettoyage · Typage |
| Mart | `sport_metrics.mart` | Table Delta | MERGE incrémental |
| BI | Power BI | Dashboard | Visualisation |

---

### Notebooks

| # | Notebook | Description |
|---|---|---|
| 00 | setup | Infrastructure : catalogue, schémas, volume |
| 01 | raw_ingestion | CSV → Tables Delta |
| 02 | staging | Vues de nettoyage et transformation |
| 03 | intermediate | Calcul Fatigue Index |
| 04 | marts | Tables finales MERGE |
| 05 | documentation | Tags Unity Catalog · Règles qualité |

In [0]:
-- ============================================
-- Infrastructure : Catalogue principal
-- Choix : catalogue dédié par projet
-- → isolation des données, gouvernance claire
-- ============================================
CREATE CATALOG IF NOT EXISTS sport_metrics
COMMENT 'SportMetrics Platform: Modern Data Stack sur Databricks.
Pipeline : CSV → Delta Lake → dbt → Power BI.
Portabilité démontrée : BigQuery → Snowflake → Databricks.
Repo : github.com/Yohan-Charney/Sports_Metrics';

In [0]:
-- ============================================
-- Infrastructure : Schémas
-- Architecture Raw/Staging/Mart
-- → séparation des responsabilités
-- → cohérence avec dbt 
-- ============================================
CREATE SCHEMA IF NOT EXISTS sport_metrics.raw
COMMENT 'Données brutes CSV. Principe : immuabilité des données. jamais modifiées après ingestion.';

CREATE SCHEMA IF NOT EXISTS sport_metrics.staging
COMMENT 'Vues de transformation : typage, nettoyage, déduplication. Matérialisées en VIEW.';

CREATE SCHEMA IF NOT EXISTS sport_metrics.mart
COMMENT 'Tables finales. MERGE incrémental Delta Lake. Consommées par Power BI Dashboard.';

In [0]:
-- ============================================
-- Infrastructure : Volume de stockage
-- ============================================
CREATE VOLUME IF NOT EXISTS sport_metrics.raw.files
COMMENT 'Volume Unity Catalog pour fichiers CSV sources.
Chemin : /Volumes/sport_metrics/raw/files/';

In [0]:
-- ============================================
-- Vérification de l'installation
-- ============================================
SHOW SCHEMAS IN sport_metrics;